# Notebook 2 — a fused row softmax written in Triton, from scratch

Goal: write the kernel, prove it correct on awkward shapes and dtypes against
a float64 reference, then benchmark it against (a) the same softmax composed
from PyTorch eager ops and (b) the library kernel `torch.softmax`, and
explain the result. The one-row-per-program structure is the standard idea
from the Triton tutorials (adapted, not copied; see docs/REFERENCES.md); the
looped variant for rows wider than one block is this notebook's own.

How to run: Colab or Kaggle, GPU runtime, Run all. See `RUNBOOK.md`.

In [ ]:
# --- Environment record (runs first; written to results/env.json) ---
import json, os, platform, subprocess, sys, time, traceback
OUT = os.environ.get("STUDY_OUT", "/kaggle/working/compiler-study-results" if os.path.isdir("/kaggle") else "/content/compiler-study-results")
os.makedirs(OUT, exist_ok=True)

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return f"unavailable: {e}"

def save(name, obj):
    with open(os.path.join(OUT, name), "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print("wrote", os.path.join(OUT, name))

def version(mod):
    try:
        m = __import__(mod)
        return getattr(m, "__version__", "unknown")
    except Exception as e:
        return f"NOT AVAILABLE ({type(e).__name__}: {e})"

env = {
    "date_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "platform": platform.platform(), "python": sys.version.split()[0],
    "nvidia_smi": sh("nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total,power.limit --format=csv,noheader"),
    "nvcc": sh("nvcc --version | tail -1"),
    "torch": version("torch"), "triton": version("triton"),
    "jax": version("jax"), "jaxlib": version("jaxlib"), "numpy": version("numpy"),
}
try:
    import torch
    env["torch_cuda"] = torch.version.cuda
    env["torch_gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA"
except Exception as e:
    env["torch_cuda"] = f"unavailable: {e}"
try:
    import jax
    env["jax_devices"] = [str(d) for d in jax.devices()]
except Exception as e:
    env["jax_devices"] = f"unavailable: {e}"
print(json.dumps(env, indent=2))

save("nb2_env.json", env)

In [ ]:
import torch, triton, triton.language as tl, math, statistics
assert torch.cuda.is_available(), "needs a CUDA GPU"
torch.backends.cuda.matmul.allow_tf32 = False
MAX_BLOCK = 8192   # widest row handled in one block; wider rows use the looped kernel

@triton.jit
def softmax_row_kernel(x_ptr, y_ptr, n_cols, x_stride, y_stride, BLOCK: tl.constexpr):
    # One program per row; the whole row fits in BLOCK (a power of two >= n_cols).
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    m = cols < n_cols
    x = tl.load(x_ptr + row * x_stride + cols, mask=m, other=-float("inf")).to(tl.float32)
    x = x - tl.max(x, axis=0)                 # stabilize: exp of values <= 0
    e = tl.exp(x)                             # masked lanes: exp(-inf) = 0
    y = e / tl.sum(e, axis=0)
    tl.store(y_ptr + row * y_stride + cols, y.to(y_ptr.dtype.element_ty), mask=m)

@triton.jit
def softmax_row_looped_kernel(x_ptr, y_ptr, n_cols, x_stride, y_stride, BLOCK: tl.constexpr):
    # Rows wider than one block. Pass 1 keeps, per lane, a running max m and a
    # running sum s of exp(x - m) rescaled whenever m grows (online softmax);
    # the lanes are combined after the loop. Pass 2 writes exp(x - max) / sum.
    # Per-lane [BLOCK] accumulators keep loop-carried types consistent.
    row = tl.program_id(0)
    m_vec = tl.full([BLOCK], -float("inf"), tl.float32)
    s_vec = tl.zeros([BLOCK], tl.float32)
    for start in range(0, n_cols, BLOCK):
        cols = start + tl.arange(0, BLOCK)
        x = tl.load(x_ptr + row * x_stride + cols, mask=cols < n_cols, other=-float("inf")).to(tl.float32)
        m_new = tl.maximum(m_vec, x)
        # Guards: (-inf) - (-inf) is NaN, so lanes that are still -inf contribute 0.
        alpha = tl.where(m_vec == m_new, 1.0, tl.exp(m_vec - m_new))
        term = tl.where(x == -float("inf"), 0.0, tl.exp(x - m_new))
        s_vec = s_vec * alpha + term
        m_vec = m_new
    row_max = tl.max(m_vec, axis=0)
    scale = tl.where(m_vec == -float("inf"), 0.0, tl.exp(m_vec - row_max))
    row_sum = tl.sum(s_vec * scale, axis=0)
    for start in range(0, n_cols, BLOCK):
        cols = start + tl.arange(0, BLOCK)
        m = cols < n_cols
        x = tl.load(x_ptr + row * x_stride + cols, mask=m, other=-float("inf")).to(tl.float32)
        tl.store(y_ptr + row * y_stride + cols, (tl.exp(x - row_max) / row_sum).to(y_ptr.dtype.element_ty), mask=m)

def triton_softmax(x):
    assert x.dim() == 2 and x.is_cuda and x.stride(1) == 1
    rows, n = x.shape
    y = torch.empty_like(x)
    if n <= MAX_BLOCK:
        BLOCK = triton.next_power_of_2(max(n, 1))
        softmax_row_kernel[(rows,)](x, y, n, x.stride(0), y.stride(0), BLOCK=BLOCK, num_warps=4 if BLOCK <= 2048 else 8)
    else:
        softmax_row_looped_kernel[(rows,)](x, y, n, x.stride(0), y.stride(0), BLOCK=4096, num_warps=8)
    return y

def composed_softmax(x):          # the same math as separate eager PyTorch ops
    xf = x.float()
    e = torch.exp(xf - xf.max(dim=-1, keepdim=True).values)
    return (e / e.sum(dim=-1, keepdim=True)).to(x.dtype)

In [ ]:
# --- Correctness gate: awkward shapes, three dtypes, extreme values, against float64 ---
# Tolerance: compare in float32 after casting; one output ulp of the dtype
# (2^-23 fp32 is too tight for exp/sum reordering, so fp32 uses 1e-6 abs + 1e-5 rel;
# fp16 2^-10, bf16 2^-7 relative), justified by a single rounding of the output.
TOL = {torch.float32: (1e-6, 1e-5), torch.float16: (1e-4, 2 ** -10), torch.bfloat16: (1e-3, 2 ** -7)}
cases = []
g = torch.Generator(device="cuda").manual_seed(0)
for dtype in (torch.float32, torch.float16, torch.bfloat16):
    for rows, n in [(1, 1), (1, 3), (7, 127), (5, 1000), (3, 1023), (4, 1024), (2, 4097), (3, 8192), (2, 8193), (2, 30001)]:
        for kind in ("randn", "large", "neg_inf_lanes"):
            x = torch.randn(rows, n, device="cuda", generator=g)
            if kind == "large":
                x = x * 80.0          # exp overflows without max subtraction
            if kind == "neg_inf_lanes" and n > 1:
                x[:, ::3] = -float("inf")
            x = x.to(dtype)
            ref = torch.softmax(x.double(), dim=-1)
            out = triton_softmax(x).double()
            atol, rtol = TOL[dtype]
            err = (out - ref).abs()
            ok = bool((err <= atol + rtol * ref.abs()).all()) and bool(torch.isfinite(out).all())
            cases.append({"dtype": str(dtype), "rows": rows, "n_cols": n, "kind": kind, "ok": ok,
                          "max_abs_err": float(err.max()), "path": "single" if n <= MAX_BLOCK else "looped"})
bad = [c for c in cases if not c["ok"]]
print(f"{len(cases) - len(bad)}/{len(cases)} cases pass")
for c in bad: print("FAIL", c)
save("nb2_correctness.json", {"cases": cases, "passed": len(cases) - len(bad), "total": len(cases)})
assert not bad, "fix the kernel before benchmarking"

In [ ]:
# --- Benchmark: composed eager vs torch.softmax (library) vs Triton ---
# triton.testing.do_bench uses CUDA events and returns milliseconds; we take
# quantiles (median, p20, p80) over its repetitions. Effective GB/s = one
# read + one write of the tensor / time (minimum traffic; cache effects at
# small sizes can push it above DRAM bandwidth — read it as effective).
from triton.testing import do_bench
bench = []
for dtype in (torch.float32, torch.float16):
    for rows, n in [(4096, 256), (4096, 1024), (4096, 4096), (1024, 8192), (256, 32768)]:
        x = torch.randn(rows, n, device="cuda", dtype=dtype)
        nbytes = 2 * x.numel() * x.element_size()
        rec = {"dtype": str(dtype), "rows": rows, "n_cols": n}
        for name, fn in (("composed_eager", lambda: composed_softmax(x)), ("torch_softmax", lambda: torch.softmax(x, -1)),
                         ("triton", lambda: triton_softmax(x))):
            med, p20, p80 = do_bench(fn, warmup=25, rep=200, quantiles=[0.5, 0.2, 0.8])
            rec[name] = {"median_ms": med, "p20_ms": p20, "p80_ms": p80, "effective_gbps": nbytes / (med * 1e-3) / 1e9}
        bench.append(rec)
        print(f"{str(dtype):14s} {rows:5d}x{n:<6d}  composed {rec['composed_eager']['median_ms']:.4f}  "
              f"torch {rec['torch_softmax']['median_ms']:.4f}  triton {rec['triton']['median_ms']:.4f} ms")
save("nb2_bench.json", {"method": "triton.testing.do_bench, warmup=25 ms, rep=200 ms, median/p20/p80", "rows": bench})

In [ ]:
# --- Repeat once and check drift (kern/llm-bringup discipline: flag > 5%) ---
drift = []
for rec in bench:
    dtype = torch.float32 if "float32" in rec["dtype"] else torch.float16
    x = torch.randn(rec["rows"], rec["n_cols"], device="cuda", dtype=dtype)
    for name, fn in (("torch_softmax", lambda: torch.softmax(x, -1)), ("triton", lambda: triton_softmax(x))):
        again = do_bench(fn, warmup=25, rep=200, quantiles=[0.5])
        again = again[0] if isinstance(again, (list, tuple)) else again
        d = 100 * abs(again - rec[name]["median_ms"]) / max(again, rec[name]["median_ms"])
        drift.append({"dtype": rec["dtype"], "rows": rec["rows"], "n_cols": rec["n_cols"], "impl": name,
                      "run1_ms": rec[name]["median_ms"], "run2_ms": again, "drift_pct": d, "flagged": d > 5})
print(sum(d["flagged"] for d in drift), "of", len(drift), "cells drift > 5%")
save("nb2_drift.json", drift)

## Explanation (fill in after running, from nb2_bench.json only)

NOT RUN yet. Expected shape of the argument, to be confirmed or refuted by
the numbers: softmax is memory-bound; the composed eager version launches
several kernels and moves the tensor through memory several times (max,
subtract, exp, sum, divide, casts), the fused Triton kernel reads once and
writes once, and the library kernel is also fused. Report where Triton
matches, beats, or loses to `torch.softmax`, and at which widths the looped
path costs a second read.

In [ ]:
import shutil
zip_path = shutil.make_archive(OUT, "zip", OUT)
print("download:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not on Colab: download the zip from the file browser (Kaggle: Output tab).")